## How to run in Colab
1. Download the dataset zip [here](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%206%20-%20Applications%20for%20Data%20Analysis/W6D5%20-%20Mini%20project/Mobile%20Price%20Classification.zip) and save it in `C:\Users\sivan\Datasets`.
2. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
3. Run the first code cell and, when it asks, choose `Mobile Price Classification.zip` from `C:\Users\sivan\Datasets`.

**Run the cells from top to bottom** (Shift+Enter runs one cell).

In [ ]:
from google.colab import files
# colab tool for uploading files from my computer
uploaded = files.upload()
# opens a window where I choose the zip
!unzip -o "Mobile Price Classification.zip"
# unzips it, train.csv is inside "Mobile Price Classification/Mobile Price Classification/"

# Week 3 Day 3 – Daily Challenge: Comprehensive Mobile Price Analysis

**What I learn here**
- Master advanced data manipulation and statistical analysis using NumPy and SciPy.
- Gain skill in data preprocessing and exploration with Pandas.
- Create detailed, informative visualizations with Matplotlib.
- Turn complex analytical findings into clear, actionable insights.

In this challenge I analyze the Mobile Price Classification train dataset: 20 phone features and the target `price_range` (0 = low cost, 1 = medium, 2 = high, 3 = very high).

**Phases**
1. Data Loading and Exploration
2. Data Cleaning and Preprocessing
3. Statistical Analysis with NumPy and SciPy
4. Data Visualization with Matplotlib
5. Insight Synthesis and Conclusion

In [ ]:
import numpy as np
# numpy for math and arrays
import pandas as pd
# pandas for the table
import matplotlib.pyplot as plt
# matplotlib for charts
import seaborn as sns
# seaborn for the heatmap and box plots
from scipy import stats
# scipy.stats for the statistics and tests

## 1. Data Loading and Exploration
- Load the dataset with Pandas and explore its structure.
- Summarize the features, the target variable and their data types.
- Basic descriptive statistics.

In [ ]:
df = pd.read_csv('Mobile Price Classification/Mobile Price Classification/train.csv')
print(df.shape)
df.head()

In [ ]:
df.info()
# all column types

In [ ]:
# what each column means (from the Kaggle page)
print("""battery_power: battery capacity (mAh) | blue: has bluetooth | clock_speed: CPU speed (GHz)
dual_sim: has 2 sim slots | fc: front camera (MP) | four_g: has 4G | int_memory: storage (GB)
m_dep: thickness (cm) | mobile_wt: weight (g) | n_cores: CPU cores | pc: main camera (MP)
px_height / px_width: screen resolution (pixels) | ram: RAM (MB) | sc_h / sc_w: screen size (cm)
talk_time: battery talk time (hours) | three_g, touch_screen, wifi: yes/no features
price_range (target): 0 low, 1 medium, 2 high, 3 very high""")
print(df['price_range'].value_counts().sort_index())
# how many phones in each price group

In [ ]:
df.describe().T.round(2)
# .T so every feature is one row

**What I found:**
- 2000 phones and 21 columns: 20 features and the target `price_range`.
- All columns are numbers: 19 `int64` and 2 `float64` (`clock_speed` and `m_dep`).
- The target is perfectly balanced: 500 phones in each of the 4 price groups. This is important, because no group can dominate the tests.
- `describe` shows two strange minimums: `px_height` and `sc_w` both have a minimum of 0, which is not possible for a real screen. I handle them in the cleaning step.

## 2. Data Cleaning and Preprocessing
- Address missing or null values.
- Transform categorical data into numerical format.

In [ ]:
print(df.isnull().sum().sum(), 'missing values')
print(df.duplicated().sum(), 'duplicated rows')

In [ ]:
# the yes/no columns are already 0 and 1
binary_cols = ['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']
for col in binary_cols:
    print(col, sorted(df[col].unique()))

In [ ]:
# a screen with 0 pixels height or 0 cm width is not possible
print('px_height == 0:', (df['px_height'] == 0).sum())
print('sc_w == 0:', (df['sc_w'] == 0).sum())
# I replace those zeros with the median of the other phones
for col in ['px_height', 'sc_w']:
    median_value = df.loc[df[col] > 0, col].median()
    df[col] = df[col].replace(0, median_value)
    print(col, 'zeros replaced with', median_value)

**What I found and why it matters:**
- There are no missing values and no duplicated rows, so I do not need to fill or drop anything.
- The categorical columns (`blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen`, `wifi`) are already 0/1 numbers, so they are already in numeric format and I do not need encoding. `price_range` is also already a number from 0 to 3.
- **Suspicious zeros:** 2 phones have `px_height` = 0 and 180 phones have `sc_w` = 0 (screen width 0 cm). These are most likely missing values that were saved as 0. I replaced them with the median of the real values (565.5 and 6.0). I chose the median because it is not affected by extreme values, and I did not delete 180 rows (9% of the data). `fc` = 0 and `pc` = 0 are fine, because a phone can have no camera.

## 3. Statistical Analysis with NumPy and SciPy
### 3.1 Central tendency, variability and shape of every feature

In [ ]:
features = df.columns.drop('price_range')
rows = []
for col in features:
    values = df[col].values
    rows.append({
        'feature': col,
        'mean': np.mean(values),
        'median': np.median(values),
        'mode': stats.mode(values).mode,
        # the most common value
        'range': np.ptp(values),
        # max minus min
        'variance': np.var(values, ddof=1),
        'std': np.std(values, ddof=1),
        # ddof=1 is the sample version, same as pandas
        'skew': stats.skew(values),
        'kurtosis': stats.kurtosis(values),
        # 0 = like a normal curve, negative = flat
    })
feature_stats = pd.DataFrame(rows).set_index('feature')
feature_stats.round(2)

**What I found:**
- For almost every feature the mean and median are almost the same and the skew is close to 0. So the features are symmetric.
- The kurtosis is about -1.2 for most continuous features (battery_power, ram, px_width, int_memory, mobile_wt, talk_time). A negative kurtosis means a flat distribution, close to a **uniform** one, not a bell curve. The yes/no columns have kurtosis -2, the flattest possible.
- `fc` (front camera) is the most skewed feature (skew 1.02): most phones have a small front camera and a few have a big one. `px_height` and `sc_w` are also right skewed (about 0.7).
- `ram` has the biggest variance by far (range 3742 MB). The mode is not very useful for continuous columns like ram, because almost every value appears only once or twice.

### 3.2 Hypothesis testing between price ranges
For every feature I run a one-way ANOVA (`f_oneway`) across the 4 price groups. H0: the feature has the same mean in all 4 groups.

In [ ]:
anova_rows = []
for col in features:
    groups = [df.loc[df['price_range'] == p, col] for p in range(4)]
    # one group of values per price range
    f_stat, p_value = stats.f_oneway(*groups)
    anova_rows.append({'feature': col, 'F': f_stat, 'p_value': p_value, 'significant': p_value < 0.05})
anova = pd.DataFrame(anova_rows).set_index('feature').sort_values('F', ascending=False)
anova.round(4)

In [ ]:
# t-test: RAM of the cheapest vs the most expensive phones
ram_low = df.loc[df['price_range'] == 0, 'ram']
ram_high = df.loc[df['price_range'] == 3, 'ram']
print('Mean RAM low:', round(ram_low.mean()), '| very high:', round(ram_high.mean()))
t_stat, p_value = stats.ttest_ind(ram_low, ram_high, equal_var=False)
print('t-statistic:', round(t_stat, 2), '| p-value:', p_value)

**What I found:**
- `ram` is in a class of its own: F = 3520, while the next one, `battery_power`, has F = 32.
- 7 features have p < 0.05: ram, battery_power, px_width, px_height, mobile_wt, int_memory and n_cores. The last three are only just under 0.05, so their effect is very small.
- The other 13 features (cameras, screen size, clock speed and all the yes/no features) show no real difference between the price groups.
- **t-test:** the cheapest phones have about 785 MB RAM on average and the most expensive about 3449 MB. t = -111.37 and p is about 0, so I reject H0: the RAM is clearly different.

### 3.3 Feature-target correlations
`pearsonr` measures a straight-line relationship, and `spearmanr` uses the ranks, which suits `price_range` because it is an ordered class (0 to 3).

In [ ]:
corr_rows = []
for col in features:
    r, p_r = stats.pearsonr(df[col], df['price_range'])
    rho, p_rho = stats.spearmanr(df[col], df['price_range'])
    corr_rows.append({'feature': col, 'pearson_r': r, 'pearson_p': p_r, 'spearman_rho': rho, 'spearman_p': p_rho})
target_corr = pd.DataFrame(corr_rows).set_index('feature').sort_values('pearson_r', ascending=False)
target_corr.round(4)

**What I found:** Pearson and Spearman give almost the same numbers, so the relationships are simple and close to straight lines. `ram` has a very strong correlation with the price (r = 0.92). After that come `battery_power` (0.20), `px_width` (0.17) and `px_height` (0.15), which are weak but significant. `int_memory` (0.04) is just under p = 0.05. Every other feature has r close to 0.

### 3.4 Advanced SciPy functions
- `kruskal` (Kruskal-Wallis) is like ANOVA but uses ranks, so it does not need normal data. Many features here are flat (uniform), so it is a good check.
- `chi2_contingency` tests if a yes/no feature and the price range are independent.
- `shapiro` checks if the features are normal at all.

In [ ]:
kruskal_rows = []
for col in features:
    groups = [df.loc[df['price_range'] == p, col] for p in range(4)]
    h_stat, p_value = stats.kruskal(*groups)
    kruskal_rows.append({'feature': col, 'H': h_stat, 'p_value': p_value})
kruskal = pd.DataFrame(kruskal_rows).set_index('feature').sort_values('H', ascending=False)
print(kruskal.round(4).head(8))

In [ ]:
# chi-square: is each yes/no feature linked to the price range?
for col in binary_cols:
    table = pd.crosstab(df[col], df['price_range'])
    # counts of 0/1 in every price group
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    print(col, '| chi2 =', round(chi2, 2), '| p-value =', round(p_value, 4))

In [ ]:
# Shapiro-Wilk normality test on a few continuous features
for col in ['ram', 'battery_power', 'px_height', 'fc']:
    w_stat, p_value = stats.shapiro(df[col])
    print(col, '| W =', round(w_stat, 3), '| p-value =', p_value)

**What I found:**
- **Kruskal-Wallis** gives the same ranking as ANOVA (ram H = 1681, then battery_power, px_width, px_height). So the ANOVA results hold even though the data is not normal.
- **Chi-square:** none of the 6 yes/no features is linked to the price range (all p-values between 0.27 and 0.84). Surprisingly, having 4G, wifi or a touch screen does not make a phone more expensive in this dataset.
- **Shapiro-Wilk:** all 4 tested features have p far below 0.05, so they are not normal. This fits the negative kurtosis (flat distributions), and it is why I added the rank-based Kruskal-Wallis test.

## 4. Data Visualization with Matplotlib
### 4.1 Histograms

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), ['ram', 'battery_power', 'px_height', 'px_width', 'fc', 'clock_speed']):
    ax.hist(df[col], bins=30, edgecolor='black')
    ax.set_title('Distribution of ' + col)
    ax.set_xlabel(col)
    ax.set_ylabel('Number of phones')
plt.tight_layout()
plt.show()

**What I found:** the histograms confirm the stats table. `ram`, `battery_power`, `px_width` and `clock_speed` are almost flat (uniform): every value range has about the same number of phones. `px_height` and `fc` fall off to the right, and `fc` has a big bar at 0 (474 phones without a front camera). This looks like generated data, not a real phone market, where values usually group around typical models.

### 4.2 Scatter plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc = axes[0].scatter(df['battery_power'], df['ram'], c=df['price_range'], cmap='viridis', s=10)
# c= colors each dot by its price range
axes[0].set_title('RAM vs Battery Power by Price Range')
axes[0].set_xlabel('battery_power (mAh)')
axes[0].set_ylabel('ram (MB)')
fig.colorbar(sc, ax=axes[0], label='price_range')
sc2 = axes[1].scatter(df['px_width'], df['px_height'], c=df['price_range'], cmap='viridis', s=10)
axes[1].set_title('Screen Resolution by Price Range')
axes[1].set_xlabel('px_width')
axes[1].set_ylabel('px_height')
fig.colorbar(sc2, ax=axes[1], label='price_range')
plt.tight_layout()
plt.show()

**What I found:** in the left chart the colors make horizontal bands. The price range changes when I go up (more RAM), not when I go to the right (more battery). That shows RAM is the main factor. Battery has only a small effect: at the edges between bands, phones with more battery are a bit more often in the higher group. The right chart shows that px_height is never bigger than px_width, and the colors are mixed everywhere, so resolution alone does not separate the prices well.

### 4.3 Box plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['ram', 'battery_power', 'px_width']):
    sns.boxplot(x='price_range', y=col, data=df, ax=ax)
    ax.set_title(col + ' by Price Range')
    ax.set_xlabel('price_range (0 = low, 3 = very high)')
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

**What I found:** the RAM boxes go up in clear steps with almost no overlap (median about 720, 1687, 2577 and 3510 MB for groups 0 to 3). The battery and px_width boxes overlap a lot. Only the very high group (3) is clearly higher (battery median 1450 mAh vs 1066 for the low group). So these two features help a bit, mostly to find the most expensive phones.

### 4.4 Correlation heatmap

In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, annot_kws={'size': 7})
# annot writes the number in every square
plt.title('Correlation Heatmap of All Features and price_range')
plt.show()

**What I found:** in the `price_range` row only `ram` stands out (0.92). Between the features, there are only four noticeable pairs: `fc` and `pc` (front and main camera, 0.64), `three_g` and `four_g` (0.58, because every 4G phone also has 3G), `px_height` and `px_width` (0.51) and `sc_h` and `sc_w` (about 0.5). All the other features are almost independent of each other.

## 5. Insight Synthesis and Conclusion
**Key determinants of the price range**
1. **RAM** is by far the strongest factor: correlation 0.92, ANOVA F = 3520, and the box plots almost do not overlap. RAM alone could classify most phones.
2. **Battery power** is second (r = 0.20), then **screen resolution** (`px_width` 0.17, `px_height` 0.15). They are weak but significant in every test (ANOVA, Kruskal-Wallis, Pearson, Spearman).
3. `mobile_wt`, `int_memory` and `n_cores` are only just significant in ANOVA, so their effect is very small.

**Unexpected findings**
- None of the yes/no features (bluetooth, dual sim, 3G, 4G, touch screen, wifi) is linked to the price (chi-square p > 0.27). In real life I would expect 4G and touch screens in the expensive phones.
- The cameras (`fc`, `pc`), the CPU speed and the number of cores have almost no effect on the price, even though phone ads focus on them.
- Most features are almost uniformly distributed (kurtosis about -1.2), and the 4 classes have exactly 500 phones each. This suggests the dataset is generated, not taken from a real market.
- 180 phones had a screen width of 0 cm, which is a data quality problem. I replaced it with the median.

**Why each step mattered:** the cleaning step found the impossible zeros, the stats table showed the data is not normal, so I checked the ANOVA with the rank-based Kruskal-Wallis test, and the charts confirmed with pictures what the tests said: the price range is driven mainly by RAM.